In [0]:
-- Gold #1: daily revenue, completed orders only, one row per day
CREATE OR REPLACE TABLE shopstream.core.gold_daily_revenue AS
SELECT
  DATE(order_ts) AS order_date,
  COUNT(DISTINCT order_id) AS orders,
  SUM(quantity) AS units_sold,
  ROUND(SUM(line_revenue), 2) AS revenue
FROM shopstream.core.silver_orders
WHERE status = 'completed'
GROUP BY DATE(order_ts);

SELECT * FROM shopstream.core.gold_daily_revenue ORDER BY order_date LIMIT 7;

order_date,orders,units_sold,revenue
2026-01-01,31,74,12950.85
2026-01-02,27,75,12679.49
2026-01-03,26,53,9782.87
2026-01-04,38,105,16471.5
2026-01-05,31,69,10657.15
2026-01-06,38,88,15698.11
2026-01-07,38,88,13455.36


In [0]:
-- Gold #2: category performance, what actually makes money
CREATE OR REPLACE TABLE shopstream.core.gold_category_performance AS
SELECT
  p.category,
  COUNT(DISTINCT o.order_id) AS orders,
  SUM(o.quantity) AS units_sold,
  ROUND(SUM(o.line_revenue), 2) AS revenue,
  ROUND(SUM(o.quantity * p.unit_margin), 2) AS gross_margin
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_products p USING (product_id)
WHERE o.status = 'completed'
GROUP BY p.category;

SELECT * FROM shopstream.core.gold_category_performance ORDER BY revenue DESC;

category,orders,units_sold,revenue,gross_margin
home-kitchen,1291,1935,284432.61,112616.01
fashion,1052,1556,268969.44,107491.71
books,1058,1533,261403.86,92158.44
fitness,957,1411,258136.76,108812.66
electronics,1152,1674,243726.66,104641.39
toys,915,1326,233182.84,103171.13
grocery,902,1352,220361.17,85506.2
beauty,918,1306,187268.73,71379.89


In [0]:
-- Gold #3: customer lifetime value, who the best customers are
CREATE OR REPLACE TABLE shopstream.core.gold_customer_ltv AS
SELECT
  c.customer_id,
  c.name,
  c.country,
  c.signup_channel,
  COUNT(DISTINCT o.order_id) AS lifetime_orders,
  ROUND(SUM(o.line_revenue), 2) AS lifetime_revenue
FROM shopstream.core.silver_orders o
JOIN shopstream.core.silver_customers c USING (customer_id)
WHERE o.status = 'completed'
GROUP BY c.customer_id, c.name, c.country, c.signup_channel;

SELECT * FROM shopstream.core.gold_customer_ltv ORDER BY lifetime_revenue DESC LIMIT 10;

customer_id,name,country,signup_channel,lifetime_orders,lifetime_revenue
C00088,Mason Brown,US,email,15,6423.06
C00900,Ananya Nair,US,referral,12,6266.62
C00370,James Nair,CA,email,8,5503.26
C00688,Ava Reddy,AE,organic,13,5496.05
C00551,Divya Garcia,IN,paid_search,11,5025.4
C00826,Nikhil Anderson,US,social,10,4978.41
C00424,Yuki Gupta,AU,social,12,4784.18
C00910,Ethan Martinez,CA,email,8,4727.22
C00568,Ethan Lopez,DE,email,8,4684.69
C00230,Ananya Nair,DE,organic,9,4678.66
